## **Knowledge Distillation on MNIST**
---


We first train a larger teacher network on MNIST using ordinary cross-entropy.
Then we train the same small student architecture in two ways:

1. **Hard-label baseline**: train on the true digit labels only  
2. **Distilled student**: train on both the true labels and the teacher's softened output distribution

We train a strong teacher first. Then we freeze it. For each training image, the student sees both the correct label and the teacher's probability distribution over all classes. The hard label tells the student what is correct. The soft target tells the student how the teacher organizes similarity between classes.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}")
print(f"cuda version : {torch.version.cuda}")

print(device)

torch version: 2.6.0+cu124
cuda version : 12.4
cuda


Define Teacher and Student networks:

In [3]:

class TeacherNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


class StudentNet(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 10)
        )

    def forward(self, x):
        return self.net(x)



The teacher is a larger FCN with more hidden units and an extra hidden layer. The student is intentionally much smaller.

In [4]:

# Get parameter count for both models.
teacher_model = TeacherNet()
student_model = StudentNet()

teacher_params = sum(p.numel() for p in teacher_model.parameters())
student_params = sum(p.numel() for p in student_model.parameters())

print(f"TeacherNet parameters: {teacher_params:,}")
print(f"StudentNet parameters: {student_params:,}")


TeacherNet parameters: 936,330
StudentNet parameters: 25,450


**We will compare three models:**

- Teacher

- Student trained normally on hard labels

- Student trained with distillation


In [ ]:
##### Declarations #####

@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return correct / total


def train_net(model, loader, epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * yb.size(0)

        avg_loss = running_loss / len(loader.dataset)
        acc = evaluate_accuracy(model, test_loader)
        print(f"Epoch {epoch:2d} | loss={avg_loss:.4f} | test_acc={acc:.4f}")

    return model


#### **Distillation Loss**

In [6]:

def distillation_loss(student_logits, teacher_logits, y_true, alpha=0.5, temperature=4.0):
    """
    alpha: weight on hard-label loss
    temperature: softmax temperature for distillation
    """
    hard_loss = F.cross_entropy(student_logits, y_true)
    student_log_probs_t = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs_t = F.softmax(teacher_logits / temperature, dim=1)

    soft_loss = F.kl_div(
        student_log_probs_t,
        teacher_probs_t,
        reduction="batchmean"
    ) * (temperature ** 2)

    return alpha * hard_loss + (1 - alpha) * soft_loss


The student learns from two sources:

- The true label via ordinary cross-entropy.
- The teacher's softened class distribution via KL divergence.
- The `temperature**2` factor is standard in KD implementations because it keeps the relative scale of the gradients sensible when temperature is used.


Distillation training loop

In [ ]:
#### Training loop code #####
def train_student_kd(student, teacher, loader, epochs=5, lr=1e-3, alpha=0.5, temperature=4.0):
    student = student.to(device)
    teacher = teacher.to(device)

    teacher.eval()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        student.train()
        running_loss = 0.0

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            with torch.no_grad():
                teacher_logits = teacher(xb)

            student_logits = student(xb)

            loss = distillation_loss(
                student_logits=student_logits,
                teacher_logits=teacher_logits,
                y_true=yb,
                alpha=alpha,
                temperature=temperature
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)

        avg_loss = running_loss / len(loader.dataset)
        acc = evaluate_accuracy(student, test_loader)
        print(f"Epoch {epoch:2d} | loss={avg_loss:.4f} | test_acc={acc:.4f}")

    return student


The teacher is frozen here. We are not updating it. We only use its logits to guide the student.

- Train teacher first
- Freeze teacher
- Train student to match labels and teacher behavior

#### **Train Teacher:**

In [8]:

epochs = 3
lr = 1e-3

teacher = TeacherNet()
teacher = train_net(teacher, train_loader, epochs=epochs, lr=lr)

teacher_acc = evaluate_accuracy(teacher, test_loader)
print(f"\nFinal teacher test accuracy: {teacher_acc:.4f}")


Epoch  1 | loss=0.3006 | test_acc=0.9654
Epoch  2 | loss=0.1146 | test_acc=0.9744
Epoch  3 | loss=0.0771 | test_acc=0.9736

Final teacher test accuracy: 0.9736


Train the student baseline on hard labels only:

In [ ]:

student_hard = StudentNet(hidden_size=20)
student_hard = train_net(student_hard, train_loader, epochs=epochs, lr=lr)

student_hard_acc = evaluate_accuracy(student_hard, test_loader)
print(f"\nFinal hard-label student test accuracy: {student_hard_acc:.4f}")


Epoch  1 | loss=0.6314 | test_acc=0.9109
Epoch  2 | loss=0.3016 | test_acc=0.9219
Epoch  3 | loss=0.2655 | test_acc=0.9295

Final hard-label student test accuracy: 0.9295


In [ ]:

student_kd = StudentNet()

student_kd = train_student_kd(
    student=student_kd,
    teacher=teacher,
    loader=train_loader,
    epochs=epochs,
    lr=lr,
    alpha=0.5,
    temperature=3.5
)

student_kd_acc = evaluate_accuracy(student_kd, test_loader)
print(f"\nFinal KD student test accuracy: {student_kd_acc:.4f}")



Epoch  1 | loss=1.9350 | test_acc=0.9095
Epoch  2 | loss=0.6633 | test_acc=0.9372
Epoch  3 | loss=0.4006 | test_acc=0.9504

Final KD student test accuracy: 0.9504


In [13]:

print(f"Teacher test accuracy : {teacher_acc:.4f}")
print(f"Student hard accuracy : {student_hard_acc:.4f}")
print(f"Student KD accuracy   : {student_kd_acc:.4f}")

Teacher test accuracy : 0.9736
Student hard accuracy : 0.9295
Student KD accuracy   : 0.9504



We saw a non-negligible improvement in holdout accuracy in the distilled model vs. hard-label trained student, but the real advantage is the increased rate of inference:



In [ ]:

from time import perf_counter

t_i = perf_counter()
acc = evaluate_accuracy(teacher, test_loader)
t_total = perf_counter() - t_i
rate = len(test_ds) / t_total   
print(f"Teacher model inference rate: {rate:,.0f} samples/sec")


Teacher model inference rate: 3,949 samples/sec


In [29]:

t_i = perf_counter()
acc = evaluate_accuracy(student_kd, test_loader)
t_total = perf_counter() - t_i
rate = len(test_ds) / t_total   
print(f"KD model inference rate: {rate:,.0f} samples/sec")


KD model inference rate: 7,039 samples/sec


In [50]:
torch.save(teacher, "teacher.pth")
torch.save(student_kd, "student-kd.pth")